# ML Practice Exam — Full Syllabus Practice
### Covers: Preprocessing · Regression · Classification · Trees/Ensembles · Clustering · MLP

**Instructions:**
- Run the **Setup** cell first before answering any question.
- Each question specifies exact parameters — match them precisely (like in a real exam).
- Difficulty flows: Easy → Medium → Hard.
- Answers should match to **4 decimal places** unless stated otherwise.
- `random_state=42` wherever applicable unless the question specifies otherwise.


## ⚙️ SETUP — Run This First!

In [70]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Reproducible synthetic "Patient Health" dataset ──────────────
np.random.seed(42)
N = 1200

age           = np.random.randint(25, 75, N).astype(float)
bmi           = np.round(np.random.normal(27, 5, N), 2)
blood_pressure= np.round(np.random.normal(120, 15, N), 1)
cholesterol   = np.round(np.random.normal(200, 40, N), 1)
glucose       = np.round(np.random.normal(100, 25, N), 1)
heart_rate    = np.round(np.random.normal(75, 12, N), 1)
creatinine    = np.round(np.random.normal(1.0, 0.3, N), 3)
hemoglobin    = np.round(np.random.normal(13.5, 1.5, N), 2)

gender       = np.random.choice(['Male','Female'], N)
smoking      = np.random.choice(['Never','Former','Current'], N, p=[0.5,0.3,0.2])
region       = np.random.choice(['North','South','East','West'], N)
risk_level   = np.random.choice(['Low','Medium','High'], N, p=[0.35,0.40,0.25])

# Target: binary classification
log_odds = (-3 + 0.03*age + 0.05*bmi + 0.01*blood_pressure
            + 0.005*cholesterol + 0.01*glucose
            + (smoking=='Current')*0.8 + (risk_level=='High')*1.2)
prob = 1/(1+np.exp(-log_odds))
disease = (np.random.rand(N) < prob).astype(int)

df = pd.DataFrame({
    'age': age, 'bmi': bmi, 'blood_pressure': blood_pressure,
    'cholesterol': cholesterol, 'glucose': glucose,
    'heart_rate': heart_rate, 'creatinine': creatinine,
    'hemoglobin': hemoglobin,
    'gender': gender, 'smoking': smoking,
    'region': region, 'risk_level': risk_level,
    'disease': disease
})

# Inject ~10% missing values in 3 columns
for col, frac in [('bmi',0.08), ('cholesterol',0.10), ('region',0.09)]:
    mask = np.random.choice([True,False], N, p=[frac, 1-frac])
    df.loc[mask, col] = np.nan

print("Dataset shape:", df.shape)
print(df.head())
print("\nMissing values per column:")

print(df.isnull().sum())
print("\nClass distribution (disease):")
print(df['disease'].value_counts())


Dataset shape: (1200, 13)
    age    bmi  blood_pressure  cholesterol  glucose  heart_rate  creatinine  \
0  63.0  27.90           132.5        194.6     90.1        82.0       1.454   
1  53.0  40.22           117.3        171.7     71.7        66.9       1.123   
2  39.0  25.99           131.6        210.7     78.7        85.5       1.070   
3  67.0    NaN           143.0          NaN     52.2        65.8       0.659   
4  32.0  35.91           133.8        207.2     97.9        67.4       0.753   

   hemoglobin  gender  smoking region risk_level  disease  
0       14.90  Female   Former   West       High        1  
1       14.26  Female  Current   East        Low        1  
2       11.98  Female    Never   West       High        1  
3       12.95  Female    Never   East     Medium        1  
4       14.73  Female   Former  South     Medium        1  

Missing values per column:
age                 0
bmi                92
blood_pressure      0
cholesterol       102
glucose          

### Q1 [Easy] — Dataset Exploration
After running the setup cell above:
1. How many rows and columns does the dataset have?
2. List all columns that contain missing values and their respective missing counts.
3. What proportion (2 decimal places) of patients have disease = 1?

In [71]:
# Your code here
display(df.shape)
# display(df.isna().sum())
# df.apply(lambda)
display(df.apply(lambda col: print(col.name, col.isna().sum()) if col.isna().sum()>0 else print()))
print((df["disease"].value_counts()[1]/df["disease"].value_counts().sum())*100)
display(df["disease"].value_counts(normalize= True ))

(1200, 13)


bmi 92

cholesterol 102






region 109




age               None
bmi               None
blood_pressure    None
cholesterol       None
glucose           None
heart_rate        None
creatinine        None
hemoglobin        None
gender            None
smoking           None
region            None
risk_level        None
disease           None
dtype: object

95.91666666666666


disease
1    0.959167
0    0.040833
Name: proportion, dtype: float64

### Q2 [Easy] — Train/Test Split
Split the dataset into features (X) and target (y) where `disease` goes to y and the rest to X.
Then split into train/test with **80:20** ratio using **random_state=42**.

1. How many rows are in X_train?
2. How many positive cases (disease=1) are in y_test?

In [105]:
# Your code here
from sklearn.model_selection import train_test_split
X = df.drop(["disease"], axis= 1)
y = df["disease"]

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
)
display(f"+ve case in y_test are {y_val[y_val==1].count()}")
X_train.shape, X_val.shape, y_train.shape, y_val.shape

'+ve case in y_test are 233'

((960, 12), (240, 12), (960,), (240,))

### Q3 [Easy] — Simple Imputation
Use **SimpleImputer** to fill missing values:
- `bmi` → fill with **median**
- `cholesterol` → fill with **mean**
- `region` → fill with **most_frequent**

Apply **fit** only on the train set, then **transform** both train and test.

1. What is the median value (rounded to 2 decimal places) used to fill `bmi`?
2. What is the mean value (rounded to 2 decimal places) used to fill `cholesterol`?
3. What is the most frequent value used to fill `region`?

In [106]:
# Your code here
cols = ["bmi", "cholesterol", "region"]

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# X_train["bmi"] = SimpleImputer(strategy="median").fit_transform(X_train[["bmi"]]) 
# X_val["bmi"] = SimpleImputer(strategy="median").transform(X_val[["bmi"]]) 

# impute = Pipeline([
#     ('bmi', SimpleImputer(strategy="median")),
#     ('chols', SimpleImputer(strategy="mean")),
#     ('region', SimpleImputer(strategy="most_frequent"))
# ])
#coltrans = (name, transformer, columns)
process = ColumnTransformer(transformers=[
    ("bmi", SimpleImputer(strategy="median"), ["bmi"]),
    ("cholesterol", SimpleImputer(strategy="mean"), ["cholesterol"]),
    ("region", SimpleImputer(strategy="most_frequent"), ["region"]),
])

In [107]:
process.fit_transform(X_train[cols])
process.transform(X_val[cols])

array([[26.85, 193.6, 'East'],
       [22.9, 177.8, 'West'],
       [26.85, 231.2, 'West'],
       [28.18, 232.8, 'North'],
       [28.45, 252.3, 'South'],
       [30.3, 169.4, 'South'],
       [26.85, 200.9, 'East'],
       [32.09, 195.8, 'North'],
       [28.06, 201.5862343572241, 'South'],
       [20.0, 84.0, 'East'],
       [31.19, 205.7, 'South'],
       [23.61, 190.7, 'West'],
       [27.78, 181.1, 'South'],
       [26.85, 166.7, 'South'],
       [22.75, 148.1, 'East'],
       [29.23, 223.5, 'East'],
       [18.78, 185.6, 'West'],
       [31.0, 141.5, 'East'],
       [38.5, 190.2, 'West'],
       [26.85, 100.1, 'West'],
       [31.94, 255.2, 'West'],
       [26.99, 176.5, 'South'],
       [24.81, 200.6, 'East'],
       [28.87, 166.4, 'South'],
       [26.85, 178.1, 'North'],
       [27.02, 130.5, 'West'],
       [32.83, 241.9, 'North'],
       [26.85, 154.7, 'North'],
       [26.85, 169.2, 'South'],
       [29.62, 204.4, 'North'],
       [24.1, 190.6, 'West'],
       [30.41, 194.

In [108]:
process.named_transformers_["bmi"].statistics_
process.named_transformers_["cholesterol"].statistics_

array([201.58623436])

In [109]:
display(df["bmi"].median(),
df["cholesterol"].mean(),
df["region"].mode())

np.float64(26.925)

np.float64(200.76785063752274)

0    South
Name: region, dtype: str

In [110]:
X_train.isna().sum().sum(), X_val.isna().sum().sum()

(np.int64(238), np.int64(65))

### Q4 [Easy] — Feature Scaling
Define `num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']`.

After imputing missing values (Q3 style — use median for bmi, mean for cholesterol), apply the following on **numerical columns only** using the **train** set to fit and **test** set to transform:

- **MinMaxScaler** → store as `X_train_mm`, `X_test_mm`
- **StandardScaler** → store as `X_train_ss`, `X_test_ss`

Answer:
1. What is the **minimum** value in the MinMax-scaled `age` column of X_train? (should be 0.0)
2. What is the **mean** of StandardScaler-scaled `glucose` in X_train? (should be ~0.0 — give to 4 decimal places)
3. What is the **max** absolute value in StandardScaler-scaled `glucose` of X_test?

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.impute import SimpleImputer
import pandas as pd

num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']

# First impute, then scale (on num cols only)
imp = SimpleImputer(strategy='median')
X_train_num = imp.fit_transform(X_train[num_cols])
X_val_num  = imp.transform(X_val[num_cols])

X_train_num = pd.DataFrame(X_train_num, columns=num_cols)
X_val_num  = pd.DataFrame(X_val_num,  columns=num_cols)

mm  = MinMaxScaler()
X_train_mm = pd.DataFrame(mm.fit_transform(X_train_num), columns=num_cols)
X_val_mm  = pd.DataFrame(mm.transform(X_val_num),      columns=num_cols)

ss  = StandardScaler()
X_train_ss = pd.DataFrame(ss.fit_transform(X_train_num), columns=num_cols)
X_val_ss  = pd.DataFrame(ss.transform(X_val_num),      columns=num_cols)




MinMax age min in X_train: 0.0
StandardScaler glucose mean in X_train: -0.0
Max abs StandardScaler glucose in X_val: 3.6096


In [ ]:
print("MinMax age min in X_train:", X_train_mm['age'].min())
print("StandardScaler glucose mean in X_train:", round(X_train_ss['glucose'].mean(), 4))
print("Max abs StandardScaler glucose in X_val:", round(X_val_ss['glucose'].abs().max(), 4))

MinMax age min in X_train: 0.0
StandardScaler glucose mean in X_train: 0
Max abs StandardScaler glucose in X_val: 3.6096


### Q5 [Easy-Medium] — Encoding Categorical Variables
Use the following encoders on the **training set** (fit only on train, transform both):

- **OrdinalEncoder** on `risk_level` with explicit category order: `['Low', 'Medium', 'High']`
- **OneHotEncoder** (drop='first') on `gender` and `smoking`
- **pd.get_dummies** (drop_first=True) on `region` — apply to full df for reference only

Answer:
1. What integer does OrdinalEncoder assign to 'High'?
2. How many new columns does OneHotEncoder create for `gender` and `smoking` combined (with drop='first')?

In [121]:
X_train.dtypes

age               float64
bmi               float64
blood_pressure    float64
cholesterol       float64
glucose           float64
heart_rate        float64
creatinine        float64
hemoglobin        float64
gender                str
smoking               str
region                str
risk_level            str
dtype: object

In [122]:
nums = [i for i in X_train.select_dtypes(["float64", "int" ])]
cat = [i for i in X_train.select_dtypes(["str" ])]

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

encode = ColumnTransformer(transformers=[
    ("nums", SimpleImputer(strategy="median"), nums),
    ("cat", SimpleImputer(strategy="most_frequent"), cat),
    ("ordinal", OrdinalEncoder(categories=[['Low', 'Medium', 'High']]), ["risk_level"]),
    ("onehot", OneHotEncoder(drop="first"), ["gender", "smoking"])
])

encode.fit_transform(X_train)

In [153]:
oe = OrdinalEncoder(categories=[['Low', 'Medium', 'High']])
ord_train= oe.fit_transform(X_train[["risk_level"]])


In [158]:
oe.categories_

[array(['Low', 'Medium', 'High'], dtype=object)]

### Q6 [Medium] — ColumnTransformer Full Preprocessing Pipeline
Build a **ColumnTransformer** that:
- `num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']`:
  → Impute with **median** then scale with **StandardScaler** (use a Pipeline per step)
- `cat_ohe_cols = ['gender','smoking','region']`:
  → Impute with **most_frequent** then apply **OneHotEncoder(handle_unknown='ignore')**
- `cat_ord_col = ['risk_level']`:
  → **OrdinalEncoder** with order `['Low','Medium','High']`

Use `remainder='drop'` and `verbose_feature_names_out=False`.

Fit on X_train, transform both. Answer:
1. How many total features does the transformed X_train have?
2. What is the transformed value of `age` for the first row of X_train (StandardScaler)? Round to 4 decimal places.

In [159]:
# Your code here
num_cols = ['age','bmi','blood_pressure','cholesterol','glucose','heart_rate','creatinine','hemoglobin']
cat_ohe_cols = ['gender','smoking','region']

nums = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])
cat_ohe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown='ignore'))
])

cat_ord = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=[['Low', 'Medium', 'High']]))
])

processor = ColumnTransformer(transformers=[
    ("num", nums, num_cols),
    ("ohe", cat_ohe, cat_ohe_cols),
    ("ord", cat_ord, ["risk_level"])
], remainder="drop", verbose_feature_names_out=False)

In [181]:
X_train_ct = processor.fit_transform(X_train)

In [272]:
X_val_ct = processor.transform(X_val)

In [184]:
print("num_features: ",X_train_ct.shape)
X_train_ct[0,0]

num_features:  (960, 18)


np.float64(-0.9665600830070815)

In [338]:
processor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

### Q7 [Medium] — KBinsDiscretizer & PolynomialFeatures
**Part A — KBinsDiscretizer:**
Apply `KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')` to the `age` column of X_train (after median imputation).
1. What are the bin edges (rounded to 2 decimal places)?

**Part B — PolynomialFeatures:**
Apply `PolynomialFeatures(degree=2, include_bias=False, interaction_only=False)` to a (n×3) array formed by the first 3 numerical columns (`age`, `bmi`, `blood_pressure`) after StandardScaler.
2. How many output features are produced?

In [186]:
# Your code here
from sklearn.preprocessing import KBinsDiscretizer

kbin = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("k_bins", KBinsDiscretizer(n_bins=4, encode="ordinal", strategy='quantile'))
])
kbin.fit_transform(X_train[["age"]])

array([[0.],
       [0.],
       [1.],
       [1.],
       [0.],
       [3.],
       [0.],
       [2.],
       [3.],
       [2.],
       [1.],
       [0.],
       [3.],
       [0.],
       [0.],
       [0.],
       [1.],
       [1.],
       [2.],
       [2.],
       [1.],
       [0.],
       [3.],
       [1.],
       [1.],
       [2.],
       [1.],
       [0.],
       [3.],
       [0.],
       [0.],
       [1.],
       [0.],
       [1.],
       [1.],
       [0.],
       [2.],
       [1.],
       [1.],
       [3.],
       [0.],
       [0.],
       [3.],
       [2.],
       [1.],
       [0.],
       [3.],
       [0.],
       [3.],
       [3.],
       [3.],
       [3.],
       [3.],
       [3.],
       [0.],
       [0.],
       [3.],
       [0.],
       [1.],
       [0.],
       [3.],
       [0.],
       [2.],
       [0.],
       [0.],
       [1.],
       [2.],
       [0.],
       [3.],
       [3.],
       [3.],
       [2.],
       [2.],
       [2.],
       [0.],
       [0.],
       [2.],

In [191]:
kbin.named_steps["k_bins"].bin_edges_

array([array([25.  , 37.75, 50.  , 61.  , 74.  ])], dtype=object)

In [197]:
from sklearn.preprocessing import PolynomialFeatures
pf = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("pf", PolynomialFeatures(degree=2, include_bias=False, interaction_only=False))
])
cols = ['age', 'bmi', 'blood_pressure']
three = pf.fit_transform(X_train[cols])

In [209]:
# pf.named_steps["pf"]
pf.get_feature_names_out(['age', 'bmi', 'blood_pressure'])

array(['age', 'bmi', 'blood_pressure', 'age^2', 'age bmi',
       'age blood_pressure', 'bmi^2', 'bmi blood_pressure',
       'blood_pressure^2'], dtype=object)

### Q8 [Medium] — Feature Selection
Using the preprocessed data from Q6 (`X_train_pp`, `X_test_pp`, `y_train`, `y_test`):

**Part A — VarianceThreshold:**
Apply `VarianceThreshold(threshold=0.1)`. How many features are **removed**?

**Part B — SelectKBest:**
Apply `SelectKBest(f_classif, k=6)` on `X_train_pp`.
1. What are the indices of the 6 selected features?
2. What is the highest F-statistic score (rounded to 2 decimal places)?

**Part C — SelectPercentile:**
Apply `SelectPercentile(f_classif, percentile=40)`.
How many features are selected?

In [210]:
from sklearn.feature_selection import VarianceThreshold
# X_train_ct
vt = VarianceThreshold(threshold=0.1)
vt.fit_transform(X_train_ct)

array([[-0.96656008,  0.00629502, -0.35844486, ...,  0.        ,
         0.        ,  1.        ],
       [-0.89658816,  0.63350192, -1.24691471, ...,  1.        ,
         0.        ,  1.        ],
       [-0.82661624, -0.33485467, -1.09883641, ...,  1.        ,
         0.        ,  2.        ],
       ...,
       [-0.61670046,  1.14840487, -0.04882658, ...,  1.        ,
         0.        ,  1.        ],
       [ 1.13259763,  0.33049318,  1.0954149 , ...,  0.        ,
         0.        ,  0.        ],
       [-0.19686892, -1.96220229, -0.82960312, ...,  0.        ,
         0.        ,  0.        ]], shape=(960, 18))

In [233]:

# vt.get_feature_names_out()
# [vt.get_support()]


X_train_ctl = pd.DataFrame(X_train_ct, columns = processor.get_feature_names_out())
X_train_ctl

,age,bmi,blood_pressure,cholesterol,glucose,heart_rate,creatinine,hemoglobin,gender_Female,gender_Male,smoking_Current,smoking_Former,smoking_Never,region_East,region_North,region_South,region_West,risk_level
0,-0.966560,0.006295,-0.358445,-1.226388,-0.830951,0.022028,1.161698,0.185028,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1,-0.896588,0.633502,-1.246915,2.018123,1.359554,0.355168,0.095914,-0.871702,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
2,-0.826616,-0.334855,-1.098836,-0.931432,1.578605,0.561398,-0.059650,0.415587,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,2.0
3,-0.616700,0.661048,-1.354608,0.795785,-1.731492,-0.612526,0.046266,-0.167215,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,-1.596307,0.029603,0.429063,-0.538158,-0.956702,-1.675402,0.430213,0.268285,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
955,0.502850,0.317780,1.970423,0.559289,-0.749821,-0.192134,-0.847404,0.153006,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0
956,-1.736251,-1.358304,-1.280569,-0.142227,1.416345,-0.549070,0.718173,0.402778,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0
957,-0.616700,1.148405,-0.048827,1.178430,0.653725,0.743832,0.658595,1.555574,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
958,1.132598,0.330493,1.095415,-0.612562,-0.757934,0.712104,-0.665360,0.178623,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0


In [ ]:
X_train_ctl.columns[vt.get_support()]

18

In [240]:
from sklearn.feature_selection import SelectKBest, SelectPercentile, f_classif

sk = SelectKBest(score_func=f_classif, k = 6)
sk.fit_transform(X_train_ct, y_train)

array([[-0.96656008, -1.22638803, -0.83095106,  1.        ,  0.        ,
         1.        ],
       [-0.89658816,  2.01812346,  1.3595541 ,  0.        ,  1.        ,
         1.        ],
       [-0.82661624, -0.93143244,  1.57860462,  0.        ,  1.        ,
         2.        ],
       ...,
       [-0.61670046,  1.17843007,  0.65372466,  0.        ,  0.        ,
         1.        ],
       [ 1.13259763, -0.61256153, -0.75793422,  0.        ,  1.        ,
         0.        ],
       [-0.19686892,  0.19790203,  0.28864047,  0.        ,  1.        ,
         0.        ]], shape=(960, 6))

In [249]:
sk.get_feature_names_out()
sk.scores_.max()

np.float64(12.554994594355904)

In [246]:
X_train_ctl.columns[sk.get_support()]

Index(['age', 'cholesterol', 'glucose', 'smoking_Former', 'smoking_Never',
       'risk_level'],
      dtype='str')

In [251]:
sp = SelectPercentile(score_func=f_classif, percentile=40)
sp.fit_transform(X_train_ct, y_train)

array([[-0.96656008,  0.00629502, -1.22638803, ...,  1.        ,
         0.        ,  1.        ],
       [-0.89658816,  0.63350192,  2.01812346, ...,  0.        ,
         1.        ,  1.        ],
       [-0.82661624, -0.33485467, -0.93143244, ...,  0.        ,
         1.        ,  2.        ],
       ...,
       [-0.61670046,  1.14840487,  1.17843007, ...,  0.        ,
         0.        ,  1.        ],
       [ 1.13259763,  0.33049318, -0.61256153, ...,  0.        ,
         1.        ,  0.        ],
       [-0.19686892, -1.96220229,  0.19790203, ...,  0.        ,
         1.        ,  0.        ]], shape=(960, 7))

In [256]:
X_train_ctl.columns[sp.get_support()]

Index(['age', 'bmi', 'cholesterol', 'glucose', 'smoking_Former',
       'smoking_Never', 'risk_level'],
      dtype='str')

### Q9 [Medium] — RFE & Pipeline Feature Selection
**Part A — RFE:**
Use `RFE(estimator=LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=5)` on `X_train_pp` and `y_train`.
1. Which feature **indices** are selected?
2. What are the ranking values of the first 5 features?

**Part B — Pipeline with SelectKBest:**
Build a `Pipeline` with steps:
1. `SelectKBest(f_classif, k=8)`
2. `StandardScaler()`  ← (data is already scaled but this tests pipeline chaining)
3. `LogisticRegression(max_iter=500, random_state=42)`

Fit on `X_train_pp`/`y_train`, report F1-score on test set (round to 4 decimal places).

In [258]:
from sklearn.feature_selection import RFE, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
rfe = RFE(estimator= LogisticRegression(max_iter=1000, random_state=42), n_features_to_select=5)

rfe.fit_transform(X_train_ct, y_train)


array([[-0.96656008, -1.22638803, -0.83095106,  1.        ,  0.        ],
       [-0.89658816,  2.01812346,  1.3595541 ,  0.        ,  0.        ],
       [-0.82661624, -0.93143244,  1.57860462,  0.        ,  0.        ],
       ...,
       [-0.61670046,  1.17843007,  0.65372466,  0.        ,  0.        ],
       [ 1.13259763, -0.61256153, -0.75793422,  0.        ,  0.        ],
       [-0.19686892,  0.19790203,  0.28864047,  0.        ,  0.        ]],
      shape=(960, 5))

In [268]:
# X_train_ctl.columns[rfe.get_support()]
rfe.get_feature_names_out()
rfe.ranking_[:5]

array([ 1,  4, 12,  1,  1])

In [270]:
lr_process = Pipeline([
    ("feat", SelectKBest(score_func=f_classif, k = 8)),
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42))
])

lr_process.fit(X_train_ct, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('feat', ...), ('scale', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...x7bd5670d49a0>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",8
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C`

In [275]:
from sklearn.metrics import f1_score
print(f1_score(y_val, lr_process.predict(X_val_ct)))

0.985200845665962


### Q10 [Medium] — PCA
Apply `PCA(n_components=5, random_state=42)` on `X_train_pp`.

1. What is the sum of explained variance ratios of all 5 components? (round to 4 decimal places)
2. Which component index (0-based) has the highest explained variance?
3. Now apply `PCA(n_components=0.90)` — how many components are needed to explain 90% variance?

In [283]:
# Your code here
from sklearn.decomposition import PCA
pcaa = PCA(n_components=5, random_state=42)
pcaa.fit(X_train_ct)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",5
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SVD 

In [291]:
# pcaa.explained_variance_ratio_

pcaa.explained_variance_ratio_.argmax()

np.int64(0)

In [293]:
from sklearn.decomposition import PCA
p = PCA(n_components=0.9, random_state=42)
p.fit(X_train_ct)

,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",0.9
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'full', 'covariance_eigh', 'arpack', 'randomized'}, default='auto'""auto"" : The solver is selected by a default 'auto' policy is based on `X.shape` and `n_components`: if the input data has fewer than 1000 features and more than 10 times as many samples, then the ""covariance_eigh"" solver is used. Otherwise, if the input data is larger than 500x500 and the number of components to extract is lower than 80% of the smallest dimension of the data, then the more efficient ""randomized"" method is selected. Otherwise the exact ""full"" SVD is computed and optionally truncated afterwards.""full"" : Run exact full SVD calling the standard LAPACK solver via `scipy.linalg.svd` and select the components by postprocessing""covariance_eigh"" : Precompute the covariance matrix (on centered data), run a classical eigenvalue decomposition on the covariance matrix typically using LAPACK and select the components by postprocessing. This solver is very efficient for n_samples >> n_features and small n_features. It is, however, not tractable otherwise for large n_features (large memory footprint required to materialize the covariance matrix). Also note that compared to the ""full"" solver, this solver effectively doubles the condition number and is therefore less numerical stable (e.g. on input data with a large range of singular values).""arpack"" : Run SVD truncated to `n_components` calling ARPACK solver via `scipy.sparse.linalg.svds`. It requires strictly `0 < n_components < min(X.shape)`""randomized"" : Run randomized SVD by the method of Halko et al... versionadded:: 0.18.0.. versionchanged:: 1.5 Added the 'covariance_eigh' solver.",'auto'
,"tol tol: float, default=0.0Tolerance for singular values computed by svd_solver == 'arpack'.Must be of range [0.0, infinity)... versionadded:: 0.18.0",0.0
,"iterated_power iterated_power: int or 'auto', default='auto'Number of iterations for the power method computed bysvd_solver == 'randomized'.Must be of range [0, infinity)... versionadded:: 0.18.0",'auto'
,"n_oversamples n_oversamples: int, default=10This parameter is only relevant when `svd_solver=""randomized""`.It corresponds to the additional number of random vectors to sample therange of `X` so as to ensure proper conditioning. See:func:`~sklearn.utils.extmath.randomized_svd` for more details... versionadded:: 1.1",10
,"power_iteration_normalizer power_iteration_normalizer: {'auto', 'QR', 'LU', 'none'}, default='auto'Power iteration normalizer for randomized SV

In [297]:
p.n_components_

np.int64(11)

---
## 🔢 REGRESSION SECTION
### Setup for Regression Questions (Q11–Q18)
Run this cell before attempting Q11–Q18. We switch to the California Housing dataset.

In [298]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

housing = fetch_california_housing()
Xr = pd.DataFrame(housing.data, columns=housing.feature_names)
yr = pd.Series(housing.target, name='MedHouseVal')

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.20, random_state=42)

# Scale features
ss_r = StandardScaler()
Xr_train_s = ss_r.fit_transform(Xr_train)
Xr_test_s  = ss_r.transform(Xr_test)

print("Regression dataset shape:", Xr.shape)
print("Feature names:", housing.feature_names)
print("Train shape:", Xr_train_s.shape, "| Test shape:", Xr_test_s.shape)


Regression dataset shape: (20640, 8)
Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
Train shape: (16512, 8) | Test shape: (4128, 8)


### Q11 [Easy-Medium] — Linear Regression
Apply `LinearRegression()` on `Xr_train_s` / `yr_train`.

1. What is the **R² score** on the test set? (round to 4 decimal places)
2. What is the **RMSE** (Root Mean Squared Error) on the test set? (round to 4 decimal places)
3. What is the **coefficient** of the feature `MedInc`? (round to 4 decimal places)
4. What is the **intercept**? (round to 4 decimal places)

In [302]:
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(Xr_train_s, yr_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [304]:
y_preds = lr.predict(Xr_test_s)
print(r2_score(yr_test, y_preds), root_mean_squared_error(yr_test, y_preds))

0.5757877060324511 0.7455813830127761


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
14196,3.2596,33.0,5.017657,1.006421,2300.0,3.691814,32.71,-117.03
8267,3.8125,49.0,4.473545,1.041005,1314.0,1.738095,33.77,-118.16
17445,4.1563,4.0,5.645833,0.985119,915.0,2.723214,34.66,-120.48
14265,1.9425,36.0,4.002817,1.033803,1418.0,3.994366,32.69,-117.11
2271,3.5542,43.0,6.268421,1.134211,874.0,2.300000,36.78,-119.80
...,...,...,...,...,...,...,...,...
11284,6.3700,35.0,6.129032,0.926267,658.0,3.032258,33.78,-117.96
11964,3.0500,33.0,6.868597,1.269488,1753.0,3.904232,34.02,-117.43
5390,2.9344,36.0,3.986717,1.079696,1756.0,3.332068,34.03,-118.38
860,5.7192,15.0,6.395349,1.067979,1777.0,3.178891,37.58,-121.96


In [312]:
# lr.coef_[list(housing.feature_names).index("MedInc")]
lr.coef_[0]
lr.intercept_

np.float64(2.071946937378881)

### Q12 [Medium] — Ridge vs Lasso vs LinearRegression
Train all three models with default parameters (except `random_state=42` for Ridge/Lasso):

| Model | Params |
|---|---|
| `LinearRegression` | default |
| `Ridge` | `alpha=1.0, random_state=42` |
| `Lasso` | `alpha=0.1, random_state=42` |

1. Which model gives the **lowest MSE** on the test set?
2. What is the MSE of Ridge? (round to 4 decimal places)
3. How many coefficients does Lasso **zero out**?

In [ ]:
from 

### Q13 [Medium] — SGDRegressor
Train `SGDRegressor` with the following parameters:
- `loss='squared_error'`
- `penalty='l2'`
- `alpha=0.001`
- `learning_rate='invscaling'`
- `eta0=0.01`
- `max_iter=200`
- `random_state=42`
- `warm_start=False`

1. What is the R² score on the test set? (round to 4 decimal places)
2. What is the R² on the training set? (round to 4 decimal places)

In [21]:
# Your code here


### Q14 [Medium-Hard] — cross_val_score
Using `LinearRegression()` and `Xr_train_s` / `yr_train`:
- Run `cross_val_score` with `scoring='r2'`, `cv=5`

1. What is the **maximum** R² across 5 folds? (round to 4 decimal places)
2. What is the **mean** R²? (round to 4 decimal places)
3. What is the **standard deviation** of R² scores? (round to 4 decimal places)

In [326]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
lr = LinearRegression()
# lr.fit(Xr_train_s, yr_train)
# y_preds = lr.predict(Xr_test_s)

In [327]:
cvs = cross_val_score(lr, Xr_test_s, yr_test, scoring='r2', cv = 5)
print(cvs.max(), f"{cvs.mean():.4f}", cvs.std(), end= "\n")

0.6184236392345073 0.5696 0.03290339822172663


### Q15 [Hard] — Pipeline: PolynomialFeatures + Ridge + GridSearchCV
Build a `Pipeline`:
1. `PolynomialFeatures(interaction_only=True)`  ← step name: `'poly'`
2. `Ridge()`  ← step name: `'ridge'`

Then apply `GridSearchCV` with:
- `param_grid = {'poly__degree': [1, 2], 'ridge__alpha': [0.1, 1.0, 10.0, 100.0]}`
- `scoring = 'neg_mean_squared_error'`
- `cv = 5`
- `n_jobs = -1`

Fit on `Xr_train_s` / `yr_train`.

1. What is the best `degree` and `alpha`?
2. What is the R² of the best estimator on the test set? (round to 4 decimal places)

In [329]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
pipe = Pipeline([
    ("poly", PolynomialFeatures(interaction_only=True)),
    ("ridge", Ridge())
])

param_grid = {
    "poly__degree": [1,2],
    "ridge__alpha": [0.1, 1.0, 10.0, 100.0]
}

gs = GridSearchCV(
    estimator= pipe,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv = 5,
    n_jobs=-1
)

In [331]:
gs.fit(Xr_train_s, yr_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...e', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'poly__degree': [1, 2], 'ridge__alpha': [0.1, 1.0, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is 

In [ ]:
gs.best_params_.predict()

{'poly__degree': 1, 'ridge__alpha': 0.1}

### Q16 [Hard] — SelectKBest(f_regression) + Lasso Pipeline
Build a `Pipeline`:
1. `SelectKBest(f_regression, k=5)` ← step name `'skb'`
2. `Lasso(random_state=42)` ← step name `'lasso'`

Fit on `Xr_train_s` / `yr_train`.

1. What is the **maximum p-value** among all 8 features? (round to 4 decimal places)
2. What are the **indices** of the 5 selected features?
3. What is the R² score on the test set? (round to 4 decimal places)

In [24]:
# Your code here


---
## 🎯 CLASSIFICATION SECTION
### Setup for Classification Questions (Q17–Q30)
Run this cell. We return to the **Patient Health** dataset.
Use the ColumnTransformer-preprocessed data from Q6 — re-run it here for a clean state.

In [25]:
# Your code here


### Q17 [Easy-Medium] — Logistic Regression (Specific Params)
Train `LogisticRegression` with:
- `solver='lbfgs'`
- `C=0.5`
- `max_iter=300`
- `tol=1e-4`
- `random_state=42`
- `penalty='l2'`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **intercept** (bias term) of the model? (round to 4 decimal places)
3. What is the **accuracy** on the test set? (round to 4 decimal places)

In [26]:
# Your code here


### Q18 [Medium] — GridSearchCV with Logistic Regression
Use `GridSearchCV` with `LogisticRegression(random_state=42)` and the following grid:
- `solver`: `['lbfgs', 'sag']`
- `C`: `[0.01, 0.1, 1, 10]`
- `max_iter`: `[100, 200, 500]`
- `cv = 4`
- `scoring = 'f1'` (default)

1. What is the **best value of C**?
2. What is the **best solver**?
3. What is the **precision score** on the test set using the best estimator? (round to 4 decimal places)
4. What is the **recall score**? (round to 4 decimal places)

In [27]:
# Your code here


### Q19 [Medium] — SGDClassifier
Train `SGDClassifier` with:
- `loss='log_loss'`
- `penalty='l2'`
- `alpha=0.001`
- `max_iter=100`
- `learning_rate='constant'`
- `eta0=0.01`
- `random_state=42`
- `warm_start=False`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **accuracy** on the test set? (round to 4 decimal places)

In [28]:
# Your code here


### Q20 [Medium] — KNeighborsClassifier + GridSearchCV
Use `GridSearchCV` with `KNeighborsClassifier()`:
- `n_neighbors`: `[3, 5, 7, 9, 11]`
- `metric`: `['euclidean', 'manhattan']`
- `cv = 5`
- `scoring = 'f1'`

1. What is the **best K** (n_neighbors)?
2. What is the **best metric**?
3. What is the **F1-score** of the best model on the test set? (round to 4 decimal places)

In [29]:
# Your code here


### Q21 [Medium] — Naive Bayes
Train `GaussianNB()` (default params) on `X_train_pp` / `y_train`.

1. What is the **accuracy** on the test set? (round to 4 decimal places)
2. What is the **F1-score**? (round to 4 decimal places)
3. What is the **AUC-ROC** score? (round to 4 decimal places)
   Hint: use `predict_proba` to get probabilities.

In [30]:
# Your code here


### Q22 [Medium-Hard] — Support Vector Machine
Train `SVC` with:
- `kernel='rbf'`
- `C=1.0`
- `gamma='scale'`
- `random_state=42`
- `probability=True`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **AUC-ROC** score? (round to 4 decimal places)
3. Now use `GridSearchCV` with `SVC(kernel='rbf', random_state=42, probability=True)`:
   - `C`: `[0.1, 1, 10]`
   - `gamma`: `['scale', 'auto']`
   - `cv=3`, `scoring='f1'`
   What are the **best C and gamma**?

In [31]:
# Your code here


### Q23 [Medium] — Confusion Matrix Deep Dive
Using the **Logistic Regression from Q17** (refit if needed):
- Compute the **confusion matrix**
- Report: **TP, TN, FP, FN**
- Compute manually: Precision, Recall, F1 (verify against sklearn)
- What is the **False Positive Rate** (FPR = FP / (FP+TN))? (round to 4 decimal places)

In [32]:
# Your code here


---
## 🌳 TREES & ENSEMBLES SECTION (Q24–Q31)
(Continue using X_train_pp, X_test_pp, y_train, y_test from the Classification Setup cell)

In [33]:
# Your code here


### Q24 [Medium] — DecisionTreeClassifier + GridSearchCV
Use `GridSearchCV` with `DecisionTreeClassifier(random_state=42)`:
- `criterion`: `['gini', 'entropy']`
- `max_depth`: `[4, 5, 6, 7, 8]`
- `min_samples_leaf`: `[5, 10, 15, 20]`
- `min_samples_split`: `[10, 20, 30]`
- `cv = 3`

1. What are the **best hyperparameters**?
2. What is the **depth** of the best estimator?
3. How many **leaves** does the best tree have? (use `.get_n_leaves()`)
4. What is the **accuracy** on the test set? (round to 4 decimal places)

In [34]:
# Your code here


### Q25 [Medium-Hard] — Decision Tree: Two Approaches Comparison
Train two `DecisionTreeClassifier` models (both with `random_state=42`, `criterion='entropy'`):

- **Approach A**: `max_depth=8, min_samples_split=40`
- **Approach B**: `ccp_alpha=0.005` (cost complexity pruning)

1. Which approach gives **higher accuracy** on the test set?
2. What is the **absolute difference in True Positives (TP)** between both approaches?
3. What is the **sum of total nodes** (node_count) of both trees?

In [35]:
# Your code here


### Q26 [Medium-Hard] — RandomForestClassifier
Train `RandomForestClassifier` with:
- `n_estimators=100`
- `max_depth=8`
- `min_samples_leaf=5`
- `oob_score=True`
- `random_state=42`
- `n_jobs=-1`

1. What is the **OOB score**? (round to 4 decimal places)
2. What is the **F1-score** on the test set? (round to 4 decimal places)
3. Which feature has the **highest importance**? Print feature index and importance value.
4. What is the **AUC-ROC** on the test set? (round to 4 decimal places)

In [36]:
# Your code here


### Q27 [Medium-Hard] — BaggingClassifier + VotingClassifier
**Part A — Bagging:**
Train `BaggingClassifier` with:
- `estimator=DecisionTreeClassifier(random_state=42)`
- `n_estimators=50`
- `max_samples=0.8`
- `max_features=0.8`
- `oob_score=True`
- `random_state=42`

1. OOB score? (round to 4 decimal places)
2. F1-score on test? (round to 4 decimal places)

**Part B — VotingClassifier (Hard & Soft):**
Combine `LogisticRegression(max_iter=500, random_state=42)`, `GaussianNB()`, and `DecisionTreeClassifier(max_depth=5, random_state=42)`.
3. Which voting type (`'hard'` or `'soft'`) gives **higher accuracy** on the test set?

In [37]:
# Your code here


### Q28 [Hard] — AdaBoost & GradientBoosting
**Part A — AdaBoostClassifier:**
- `n_estimators=100`
- `learning_rate=0.5`
- `algorithm='SAMME'`
- `random_state=42`
- Base estimator: `DecisionTreeClassifier(max_depth=2, random_state=42)`

1. F1-score on test? (round to 4 decimal places)

**Part B — GradientBoostingClassifier with GridSearchCV:**
- `n_estimators`: `[50, 100]`
- `max_depth`: `[2, 3]`
- `learning_rate`: `[0.05, 0.1]`
- `cv=3`, `scoring='f1'`, `random_state=42`

2. What are the best params?
3. F1-score of best model on test? (round to 4 decimal places)

In [38]:
# Your code here


### Q29 [Hard] — XGBoost
Train `XGBClassifier` with:
- `n_estimators=100`
- `max_depth=4`
- `learning_rate=0.1`
- `subsample=0.8`
- `colsample_bytree=0.8`
- `use_label_encoder=False`
- `eval_metric='logloss'`
- `random_state=42`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. What is the **AUC-ROC** score? (round to 4 decimal places)
3. What is the index of the **most important feature** according to `feature_importances_`?

In [39]:
# Your code here


---
## 🔵 CLUSTERING SECTION (Q30–Q32)
For clustering, use only the numerical features of df (after imputation & scaling).
Run this setup first.

In [40]:
# Your code here


### Q30 [Medium] — KMeans Clustering
Train `KMeans` with:
- `n_clusters=3`
- `init='k-means++'`
- `n_init=10`
- `max_iter=300`
- `random_state=42`

1. What is the **inertia** (within-cluster sum of squares)? (round to 2 decimal places)
2. How many samples fall in each cluster? (sorted by cluster index)
3. Now run KMeans for `k = 2, 3, 4, 5, 6` and print the inertia for each.
   (This is the elbow method — use this to identify the optimal k visually.)

In [41]:
# Your code here


### Q31 [Medium-Hard] — Hierarchical Agglomerative Clustering
Apply `AgglomerativeClustering` with:
- `n_clusters=3`
- `linkage='ward'`
- `metric='euclidean'`

1. How many samples are in each cluster?
2. Now use `n_clusters=None, distance_threshold=5, linkage='ward'` — how many clusters are formed automatically?
3. Compute the **silhouette score** for both settings. Which gives a higher silhouette score?

In [42]:
# Your code here


---
## 🧠 MLP SECTION (Q32–Q34)
(Use X_train_pp, X_test_pp, y_train, y_test from Classification Setup)

In [43]:
# Your code here


### Q32 [Hard] — MLPClassifier (Specific Params)
Train `MLPClassifier` with:
- `hidden_layer_sizes=(64, 32)`
- `activation='relu'`
- `solver='adam'`
- `alpha=0.001`
- `batch_size=64`
- `learning_rate_init=0.001`
- `max_iter=200`
- `random_state=42`
- `early_stopping=True`
- `validation_fraction=0.1`

1. What is the **F1-score** on the test set? (round to 4 decimal places)
2. How many **iterations** did training run for? (`.n_iter_`)
3. What is the **best validation score** achieved during training? (`.best_validation_score_`, round to 4 decimal places)

In [44]:
# Your code here


### Q33 [Hard] — MLPClassifier + GridSearchCV
Use `GridSearchCV` with `MLPClassifier(random_state=42, max_iter=200)`:
- `hidden_layer_sizes`: `[(32,), (64,), (64, 32), (128, 64)]`
- `alpha`: `[0.0001, 0.001, 0.01]`
- `solver`: `['adam', 'sgd']`
- `cv = 3`, `scoring = 'f1'`

1. What are the **best hyperparameters**?
2. What is the **F1-score** of the best model on the test set? (round to 4 decimal places)

In [45]:
# Your code here


### Q34 [Hard] — MLPRegressor on California Housing
Using the regression data (`Xr_train_s`, `Xr_test_s`, `yr_train`, `yr_test`):

Train `MLPRegressor` with:
- `hidden_layer_sizes=(100, 50)`
- `activation='relu'`
- `solver='adam'`
- `alpha=0.001`
- `max_iter=300`
- `random_state=42`
- `early_stopping=True`

1. What is the **R² score** on the test set? (round to 4 decimal places)
2. What is the **RMSE** on the test set? (round to 4 decimal places)

In [46]:
# Your code here


---
## ⚖️ HANDLING IMBALANCE (Q35)
### Q35 [Hard] — SMOTE + RandomForest Pipeline
Create a synthetic **imbalanced** version of our dataset (minority class = 15%).
Then compare performance **before and after SMOTE**.

Setup:
```python
from imblearn.over_sampling import SMOTE
# Use X_train_pp, y_train but artificially make it imbalanced for demonstration
```

Steps:
1. Apply `SMOTE(random_state=42)` to `X_train_pp` / `y_train`.
2. Print class distribution **before** and **after** SMOTE.
3. Train `RandomForestClassifier(n_estimators=100, random_state=42)` on:
   - Original `X_train_pp` / `y_train`
   - SMOTE-resampled data
4. Report **F1-score** for both. Which is better?

In [47]:
# Your code here


---
## 🏆 BONUS HARD QUESTION (Q36)
### Q36 [Hard] — End-to-End Pipeline: Preprocessing → Feature Selection → Model → Hyperparameter Tuning
Build a **single end-to-end Pipeline** that:

1. **Preprocessing**: `ColumnTransformer` (as in Q6 — impute + scale numericals, OHE categoricals, Ordinal encode risk_level)
2. **Feature Selection**: `SelectKBest(f_classif)` with `k` as a hyperparameter
3. **Classifier**: `RandomForestClassifier(random_state=42)`

Use `GridSearchCV` with:
- `select__k`: `[8, 10, 12]`
- `clf__n_estimators`: `[50, 100]`
- `clf__max_depth`: `[5, 8, None]`
- `cv=3`, `scoring='f1'`, `n_jobs=-1`

Fit on raw `X_train` / `y_train` (not the preprocessed version — the pipeline handles it).

1. What are the **best hyperparameters**?
2. What is the **F1-score** of the best end-to-end pipeline on `X_test`/`y_test`? (round to 4 decimal places)
3. What is the **AUC-ROC**? (round to 4 decimal places)

In [48]:
# Your code here
